In [0]:
type_taxi = "yellow"
df = spark.table(f"ifood.landing.{type_taxi}_taxi")


In [0]:
from pyspark.sql import functions as F

if type_taxi == "green":
    df = df.withColumnRenamed("lpep_pickup_datetime", "tpep_pickup_datetime")\
        .withColumnRenamed("lpep_dropoff_datetime", "tpep_dropoff_datetime")

ERROS = {
    "0": "Sucesso",
    "1": "Não consta vendedor",
    "2": "Saida > entrada",
    "3": "Não consta passageiro",
    "4": "Distancia nula",
    "5": "Não consta pagamento",
    "6": "Valor negativo",
    "7": "Flag invalida",
}

colunas_valor = [
    "fare_amount",
    "tip_amount",
    "tolls_amount",
    "total_amount",
]

condicoes_erro = [
    (
        F.col("VendorID").isNull() | ~F.col("VendorID").isin([1, 2, 6, 7]),
        F.lit("1")
    ),
    (
        F.col("tpep_pickup_datetime").isNull() |
        F.col("tpep_dropoff_datetime").isNull() |
        (F.col("tpep_pickup_datetime") >= F.col("tpep_dropoff_datetime")),
        F.lit("2")
    ),
    (
        F.col("passenger_count").isNull() | (F.col("passenger_count") < 1),
        F.lit("3")
    ),
    (
        F.col("trip_distance").isNull() | (F.col("trip_distance") <= 0),
        F.lit("4")
    ),
    (
        F.col("payment_type").isNull() | ~F.col("payment_type").isin([0, 1, 2, 3, 4, 5, 6]),
        F.lit("5")
    ),
    (
        F.col("store_and_fwd_flag").isNull() | ~F.col("store_and_fwd_flag").isin(["Y", "N"]),
        F.lit("7")
    ),
    (
        F.array_contains(
            F.array(*[
                F.col(c).isNull() | (F.col(c) < 0)
                for c in colunas_valor
            ]),
            True
        ),
        F.lit("6")
    ),
]

erro_col = F.lit("0")

for condicao, codigo_erro in condicoes_erro:
    erro_col = F.when(
        erro_col == "0",
        F.when(condicao, codigo_erro).otherwise(erro_col)
    ).otherwise(erro_col)

df_validado = df.withColumn("erro_concat", erro_col)

df_validado = df.withColumn("erro_concat", erro_col)

df_ok = (
    df_validado
    .filter(F.col("erro_concat") == "0")
    .withColumn("origem", F.lit(type_taxi))
)

df_reject = (
    df_validado
    .filter(F.col("erro_concat") != "0")
    .select("id_data", "erro_concat")
    .withColumn("origem", F.lit(type_taxi))
)

In [0]:
trusted_path = "s3://ifood-case-data-quality/trusted-data/"+f"{type_taxi}_taxi/"
reject_path = "s3://ifood-case-data-quality/rejected-data/"

(df_ok.write.format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .partitionBy("year", "month")
    .save(trusted_path)
)
(df_reject.write \
    .format("delta") \
    .mode("append") \
    .save(reject_path)
)